In [6]:
# Load environment variables from .env file
import os
import sys
from pathlib import Path
from dotenv import load_dotenv

# Resolve project root and ensure src/ is importable
project_root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

env_path = project_root / '.env'
load_dotenv(dotenv_path=env_path)

from src.env_config import configure_runtime_env

runtime = configure_runtime_env(env_path)
print(f"✓ LangSmith tracing: {runtime['langsmith_tracing']}")
print(f"✓ SSL verify: {runtime['ssl_verify']}")

# Verify required keys are loaded
missing = [k for k in ("SERPER_API_KEY", "ANTHROPIC_API_KEY") if not os.getenv(k)]
if missing:
    print(f"⚠️ Warning: missing env vars: {', '.join(missing)}. Check .env file.")
else:
    print("✓ SERPER_API_KEY loaded successfully")
    print("✓ ANTHROPIC_API_KEY loaded successfully")
    print(f"✓ LLM provider: {os.getenv('LLM_PROVIDER', 'anthropic')}")
    print(f"✓ Model: {os.getenv('ANTHROPIC_MODEL', 'claude-sonnet-4-20250514')}")

✓ LangSmith tracing: false
✓ SSL verify: true
✓ SERPER_API_KEY loaded successfully
✓ ANTHROPIC_API_KEY loaded successfully
✓ LLM provider: anthropic
✓ Model: claude-sonnet-4-20250514


In [7]:
# AI Research Agent - Date Range Input
# Define your research date range here
start_date = "2026-05-01"
end_date = "2026-05-31"

print(f"Research period: {start_date} to {end_date}")

Research period: 2026-05-01 to 2026-05-31


In [8]:
# Optional preflight: verify external APIs before running the agent
from src.serper import search as serper_ping

try:
    serper_ping("AI news", max_results=1)
    print("✓ Serper API reachable")
except RuntimeError as exc:
    print(f"⚠️ Serper check failed: {exc}")
except Exception as exc:
    print(f"⚠️ Serper check failed: {type(exc).__name__}: {exc}")

✓ Serper API reachable


In [9]:
# Run the research agent with the specified date range
from src.researcher import run_research

results = run_research(start_date, end_date)
# print(results)

{'status': 'completed', 'research_task': 'Conduct comprehensive research on AI developments from 2026-05-01 to 2026-05-31.\n\nResearch categories:\n1. AI Model Releases - New models, updates, capabilities\n2. Tools & Frameworks - Software releases, libraries, platforms\n3. Research Papers - Notable publications, breakthroughs\n4. Company Announcements - Strategic news, partnerships, funding\n5. Events - Conferences, workshops, meetups\n\nFor each category:\n- Execute multiple searches with different angles\n- Analyze results for significance\n- Note any unexpected discoveries or trends\n- Save findings with your expert analysis\n\nReturn a summary of all significant findings and confirm which files were saved.', 'output_files': ['data/research_results/model_releases.md', 'data/research_results/events_conferences.md', 'data/research_results/tools_frameworks.md', 'data/research_results/research_papers.md', 'data/research_results/company_announcements.md'], 'files_written': ['data/researc

In [10]:
# Phase 2: Synthesize research into an HTML presentation (LLM-generated)
from src.researcher import run_synthesis_and_presentation

synthesis_results = run_synthesis_and_presentation(start_date, end_date)

status = synthesis_results.get("status", "unknown")
html_file = synthesis_results.get("html_file", "N/A")
print(f"✓ Synthesis completed! (status: {status})")
print(f"HTML presentation saved to: {html_file}")
if synthesis_results.get("message"):
    print(f"Note: {synthesis_results['message']}")

✓ Synthesis completed! (status: success)
HTML presentation saved to: data/presentations/presentation.html


In [19]:
# Preview the HTML presentation in the notebook
from IPython.display import HTML, display

html_path = project_root / 'data' / 'presentations' / 'presentation.html'
if html_path.exists():
    html_content = html_path.read_text(encoding='utf-8')
    #display(HTML(html_content))
    print(f"Open in browser: {html_path.as_uri()}")
    print("Use Print (Ctrl+P) in the browser to save as PDF.")
else:
    print(f"⚠️ Presentation not found at: {html_path}")
    print("Run Cell 4 first.")

Open in browser: file:///C:/Repos/Agent/AIResearchAgent/data/presentations/presentation.html
Use Print (Ctrl+P) in the browser to save as PDF.


In [20]:
# Results summary
print("=== Research Results ===")
for k, v in results.items():
    print(f"{k}: {v}")

print("\n=== Synthesis Results ===")
for k, v in synthesis_results.items():
    print(f"{k}: {v}")

=== Research Results ===
status: skipped
message: Using existing research files; web search skipped.
files_written: ['data\\research_results\\company_announcements.md', 'data\\research_results\\events.md', 'data\\research_results\\model_releases.md', 'data\\research_results\\research_papers.md', 'data\\research_results\\tools_frameworks.md']
output_files: ['data\\research_results\\company_announcements.md', 'data\\research_results\\events.md', 'data\\research_results\\model_releases.md', 'data\\research_results\\research_papers.md', 'data\\research_results\\tools_frameworks.md']

=== Synthesis Results ===
status: success
html_file: data\presentations\presentation.html
files_written: ['data\\presentations\\presentation.html']
message: None
